In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
df = pd.read_csv("train_original.csv")
df = df.dropna(subset=["text", "label"])  # remove empty rows

In [3]:
df["text"] = df["text"].fillna("").astype(str)

In [4]:
tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(df["text"])

In [5]:
X_words = tokenizer.texts_to_sequences(df["text"])
X_words = pad_sequences(X_words, maxlen=50, padding="post")

In [6]:
# Encode labels
encoder = LabelEncoder()
df["label_encoded"] = encoder.fit_transform(df["label"])
num_classes = len(encoder.classes_)

y = to_categorical(df["label_encoded"], num_classes=num_classes)

print("X_words shape:", X_words.shape)
print("y shape:", y.shape)
print("Classes:", encoder.classes_)

X_words shape: (49573, 50)
y shape: (49573, 7)
Classes: ['anger' 'disgust' 'fear' 'happiness' 'neutral' 'sadness' 'surprize']


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_words, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)

Train: (39658, 50) (39658, 7)
Validation: (9915, 50) (9915, 7)


In [8]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.regularizers import l2

word_input = Input(shape=(50,), name="word_input")

embedding = Embedding(input_dim=20000, output_dim=128, input_length=50)(word_input)

conv = Conv1D(filters=128, kernel_size=5, activation="relu")(embedding)
pool = GlobalMaxPooling1D()(conv)

dense = Dense(64, activation="relu", kernel_regularizer=l2(0.001))(pool)
dropout = Dropout(0.6)(dense)  # stronger dropout

output = Dense(num_classes, activation="softmax", name="output")(dropout)

model = Model(inputs=word_input, outputs=output)

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 word_input (InputLayer)     [(None, 50)]              0         
                                                                 
 embedding (Embedding)       (None, 50, 128)           2560000   
                                                                 
 conv1d (Conv1D)             (None, 46, 128)           82048     
                                                                 
 global_max_pooling1d (Glob  (None, 128)               0         
 alMaxPooling1D)                                                 
                                                                 
 dense (Dense)               (None, 64)                8256      
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                             

2025-10-27 11:30:51.781731: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
2025-10-27 11:30:51.783589: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 37670 MB memory:  -> device: 0, name: NVIDIA L40S, pci bus id: 0000:a1:00.0, compute capability: 8.9


In [9]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    batch_size=128,
    epochs=30,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)


Epoch 1/30


2025-10-27 11:31:10.097182: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 90701
W0000 00:00:1761564670.170765  613473 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564670.177913  613473 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564670.182872  613473 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564670.182942  613473 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564670.182989  613473 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564670.184277  613473 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564670.184356  613473 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564670.185499  613473 gpu_

  3/310 [..............................] - ETA: 26s - loss: 2.0189 - accuracy: 0.2083  

W0000 00:00:1761564671.693323  613469 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564671.693435  613469 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564671.693492  613469 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564671.693541  613469 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564671.693587  613469 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564671.693638  613469 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564671.693690  613469 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564671.693737  613469 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564671.693791  613469 gp

  6/310 [..............................] - ETA: 23s - loss: 1.9983 - accuracy: 0.2682

...

267/310 [========================>.....] - ETA: 0s - loss: 0.3155 - accuracy: 0.9303

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)


309/310 [============================>.] - ETA: 0s - loss: 0.3172 - accuracy: 0.9285

'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
'+ptx85' is not a recognized feature for this target (ignoring feature)
W0000 00:00:1761564699.920158  613466 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564699.920279  613466 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564699.920338  613466 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564699.920386  613466 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564699.920430  613466 gpu_timer.cc:114] Skipping the delay kernel, measu

310/310 [==============================] - 2s 5ms/step - loss: 0.3175 - accuracy: 0.9284 - val_loss: 1.9541 - val_accuracy: 0.4763


In [10]:
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

print(f"Train accuracy: {train_acc:.3f}, loss: {train_loss:.3f}")
print(f"Validation accuracy: {val_acc:.3f}, loss: {val_loss:.3f}")


W0000 00:00:1761564729.510755  613469 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564729.510860  613469 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564729.510913  613469 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564729.510959  613469 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564729.511017  613469 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564729.511067  613469 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564729.511127  613469 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564729.511183  613469 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564729.511229  613469 gp

Train accuracy: 0.673, loss: 0.981
Validation accuracy: 0.539, loss: 1.288


W0000 00:00:1761564731.735844  613467 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564731.735997  613467 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564731.736092  613467 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564731.736172  613467 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564731.736263  613467 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564731.736339  613467 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564731.736419  613467 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564731.736507  613467 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564731.736585  613467 gp

In [11]:
test = pd.read_csv("test_file.csv")

# Map labels to match training
label_mapping = {
    'HAPPY': 'happiness',
    'SAD': 'sadness',
    'OTHER': 'neutral',
    'ANGRY': 'anger',
    'SURPRISE': 'surprize',
    'HATE': 'disgust',
    'FEAR': 'fear',
}
test["label"] = test["label"].map(label_mapping)
test = test.dropna(subset=["label"])
test["text"] = test["text"].fillna("").astype(str)

X_test_words = tokenizer.texts_to_sequences(test["text"])
X_test_words = pad_sequences(X_test_words, maxlen=50, padding="post")

test["label_encoded"] = encoder.transform(test["label"])
y_test = to_categorical(test["label_encoded"], num_classes=num_classes)

test_loss, test_acc = model.evaluate(X_test_words, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.3f}, loss: {test_loss:.3f}")

Test accuracy: 0.315, loss: 1.741


W0000 00:00:1761564758.357294  613473 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564758.357387  613473 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564758.357433  613473 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564758.357478  613473 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564758.357522  613473 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564758.357573  613473 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564758.357623  613473 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564758.357668  613473 gpu_timer.cc:114] Skipping the delay kernel, measurement accuracy will be reduced
W0000 00:00:1761564758.357716  613473 gp